# Low Fidelity Neural Network
Neural Network which uses LF dataset and exploits the Low fidelity component of the MF model. The NN is tested also on the HF test set  

In [1]:
#########################     LIBRARIES     ##########################
import keras.backend as K
from keras.regularizers import l2
from keras.utils import custom_object_scope
from keras.initializers import glorot_uniform
from keras.models import load_model, save_model
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from keras.optimizers import Adam, Nadam, Adamax

from time import perf_counter
import pandas
import os
from itertools import product
import keras
import tensorflow as tf

from module_utils import * 
sys.path.append('../utils')
from Structure import *

from pathlib import Path

# path to the current notebook
current_file_path = Path().resolve()
# path to the current folder
load_context_functions(current_file_path.parent.name)

c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
# reproducibility
seed=42
np.random.seed(seed)
keras.utils.set_random_seed(seed)
tf.random.set_seed(seed)

# Data Preparation

In [2]:
file_path_LF = "../DATA_reaction/reaction_diffusion_LF_46_d75.mat"
(reaction_LF_test, U_LF_test, x_LF_test) = import_data(file_path_LF)
U_LF_test = U_LF_test[:, -1, :,12]

In [3]:
reaction_LF_test = normalization(reaction_LF_test)
x_LF_test = normalization(x_LF_test)
U_LF_test = normalization(U_LF_test)

reaction_LF_test_original=reaction_LF_test
x_LF_test_original=x_LF_test
U_LF_test_original=U_LF_test

In [4]:
NepoLF=2000
Nlf=150

In [ ]:
noise_std2=[ 0.01 ,0.005 ,0.02,0.005]
noise_std1=[0.005,0.003,0.01,0.003]

(U_LF_test,reaction_LF_test)=add_noise(noise_std1,noise_std2,reaction_LF_test,U_LF_test)

noise_std2=[ 0.0001 ,0.00005 ,0.0002,0.00005]
noise_std1=[0.0025,0.0015,0.005,0.0015]

(U_LF_test,x_LF_test)=add_noise(noise_std1,noise_std2,x_LF_test,U_LF_test.T)
U_LF_test=U_LF_test.T

In [5]:
reaction_LF_test_original=np.array(list(product(reaction_LF_test_original.flatten(), x_LF_test_original.flatten())))
reaction_LF_test_original=np.c_[reaction_LF_test_original,np.abs(np.sin(5*np.pi*reaction_LF_test_original[:, 0])) ,np.abs(np.sin(3.5*np.pi*reaction_LF_test_original[:, 1])),np.ones(reaction_LF_test_original.shape[0])]

row, col = U_LF_test_original.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_LF_test_original = U_LF_test_original.flatten()[comb]

L=np.minimum(Nlf,x_LF_test.shape[0])
permutation1 = np.random.permutation(len(reaction_LF_test))
permutation2 = np.random.permutation(len(x_LF_test))
reaction_LF = reaction_LF_test[permutation1][0:Nlf]    # problema di consistenza qui: non stai confrontando stesso numero di dati, ma è cosairreparabile se non creando nuovo dataset
x_LF = x_LF_test[permutation2][0:L]

grid1, grid2 = np.meshgrid(reaction_LF, x_LF)
reaction_LF = np.column_stack((grid1.ravel(), grid2.ravel()))
reaction_LF_test = np.array(list(product(reaction_LF_test.flatten(), x_LF_test.flatten())))


reaction_LF=np.c_[reaction_LF, np.abs(np.sin(5*np.pi*reaction_LF[:, 0])),np.abs(np.sin(3.5*np.pi*reaction_LF[:, 1])),np.ones(reaction_LF.shape[0])]
reaction_LF_test=np.c_[reaction_LF_test,np.abs(np.sin(5*np.pi*reaction_LF_test[:, 0])) ,np.abs(np.sin(3.5*np.pi*reaction_LF_test[:, 1])),np.ones(reaction_LF_test.shape[0])]


p1, p2 = np.meshgrid(permutation1[0:Nlf], permutation2[0:L])
permutation = np.column_stack((p1.ravel(), p2.ravel()))
U_train_LF = U_LF_test[permutation[:,0],permutation[:,1]]

row, col = U_LF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_LF_test = U_LF_test.flatten()[comb]

        
##

# Low Fidelity Neural Network
## Low fidelity data

In [6]:
####################    NN training and PREDICTION    #######################
name="LF"
K.clear_session()
best_params = {
                    "lr": 0.0255,
                    "kernel_init": "glorot_uniform",
                    "opt": "Adam",
                } 

print("\nLF Model:")

model= NetworkFactory.build_network("LF",params=best_params,data_train=reaction_LF,output_train=U_LF,N=NepoLF,n=Nlf,do_HPO=False,verbose=False)
ULF = model.prediction(reaction_LF_test_original)

(mse_LF,R_LF) = model.performance(reaction_LF_test_original,U_LF_test_original)

  0%|          | 0/2 [00:00<?, ?trial/s, best loss=?]

1/1 [==============================] - 0s 117ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 83ms/step 

1/1 [==============================] - 0s 85ms/step 

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 101ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 99ms/step 

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 100ms/step

1/1 [==============================] - 0s 86ms/step 

1/1 [=======================

# High fidelity data

In [ ]:
########################     PREPARATION      ##########################
# introduction of the data
file_path_HF = "../DATA_reaction/reaction_diffusion_HF.mat"
(reaction_HF_test, U_HF_test, x_HF_test) = import_data(file_path_HF)
U_HF_test = U_HF_test[:, :, 44,44]

In [ ]:
########################     NORMALIZATION  #########################
# Input
reaction_HF_test = normalization(reaction_HF_test)
x_HF_test=normalization(x_HF_test)
U_HF_test=normalization(U_HF_test)

In [ ]:

reaction_HF_test = np.array(list(product(reaction_HF_test.flatten(), x_HF_test.flatten())))
reaction_HF_test_original=np.c_[reaction_HF_test,np.abs(np.sin(5*np.pi*reaction_HF_test[:, 0])) ,np.abs(np.sin(3.5*np.pi*reaction_HF_test[:, 1])),np.ones(reaction_HF_test.shape[0])]

row, col = U_HF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_HF_test_original = U_HF_test.flatten()[comb]


In [ ]:
UHF = model.prediction(reaction_HF_test_original)
(mse_HF,R_HF) = model.performance(reaction_HF_test_original,U_HF_test_original)


1252/1252 [==============================] - 2s 2ms/step
Elapsed time:  652.2149617001414

HF Data

LF Model:
Test MSE: 0.37269667817925306
R^2: -2.308641727867254


# Add plot

In [ ]:
# Name of the folder
folder_name = "HF_models_multiparam"
folder_path = os.path.join(os.getcwd(), folder_name)

# create the folder
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder '{folder_name}' created.")
else:
    print(f"Folder '{folder_name}' already exists.")
   

os.makedirs(folder_name)

R2_HF = pandas.DataFrame({"R2_HF": [R_HF]})
R2_LF = pandas.DataFrame({"R2_LF": [R_LF]})
MSE_test_LF = pandas.DataFrame({"MSE_test_LF": [mse_LF]})
MSE_test_HF = pandas.DataFrame({"MSE_test_HF": [mse_HF]})


R2_LF.to_csv(
    "./{folder_name}/r2_HF_LF200_1000steps.txt",
    header=True,
    index=False,
    sep="\t",
    mode="a",
)
MSE_test_HF.to_csv(
    "./{folder_name}/test_mse_LF200_1000steps.txt",
    header=True,
    index=False,
    sep="\t",
    mode="a",
)
R2_HF.to_csv(
    "./{folder_name}/r2_LF_LF200_1000steps.txt",
    header=True,
    index=False,
    sep="\t",
    mode="a",
)
MSE_test_LF.to_csv(
    "./{folder_name}/mse_LF_LF200_1000steps.txt",
    header=True,
    index=False,
    sep="\t",
    mode="a",
)